# L04 — Enzyme Kinetics II

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-biotek-bio-ugm/TKBM262615_Practicals/blob/main/notebooks/04_enzyme_kinetics_II.ipynb)

**Course:** TBM263214 Komputasi Biologi (Computational Biology) · **Week 4** · **CPMK2**

Companion notebook for [`scripts/04_enzyme_kinetics_II.py`](../scripts/04_enzyme_kinetics_II.py), based on the course lecture note [`04-enzyme-kinetics-2-en.md`](https://github.com/lab-biotek-bio-ugm/TKBM262615/blob/main/lecture-notes/04-enzyme-kinetics-2-en.md).

Simulates competitive and non-competitive inhibition, then the Hill function — in its activating ("switch ON") and repressing ("switch OFF") forms — as a model of allosteric, cooperative regulation.


## Learning Objectives

After this notebook, you should be able to:
1. **Distinguish competitive from non-competitive inhibition** by their effect on the apparent $K_M$ and $V_{max}$. *(CPMK2)*
2. **Model allosteric, cooperative regulation** with the Hill equation. *(CPMK2)*
3. **Write the Hill activation and repression functions**, and relate the Hill coefficient $n$ to cooperativity. *(CPMK2)*
4. **Explain sigmoidal kinetics as a "soft switch"**, and how increasing $n$ sharpens the transition. *(CPMK2)*


In [ ]:
# Setup — Colab already ships numpy, scipy, and matplotlib, so no pip install is needed.
import numpy as np
import matplotlib.pyplot as plt


## Key Concepts

- **Competitive inhibition:** inhibitor $I$ competes with substrate for the active site — it raises the *apparent* $K_M$ but leaves $V_{max}$ unchanged (enough substrate can still outcompete the inhibitor).
- **Non-competitive inhibition:** $I$ binds a separate site and lowers the *apparent* $V_{max}$, while $K_M$ is unchanged (no amount of substrate reverses it).
- **Cooperativity / allosteric regulation:** binding of one ligand changes the enzyme's affinity for the next — captured phenomenologically by the Hill equation with Hill coefficient $n$ ($n>1$: positive cooperativity).
- **Hill function as a "soft switch":** an **activating** Hill function turns a response ON as $[S]$ rises; a **repressing** Hill function turns it OFF. Both are sigmoidal, and the transition sharpens toward a step function as $n$ increases.

## Key Equations

- Competitive inhibition: $\displaystyle v = \frac{V_{max}[S]}{K_M(1+[I]/K_i) + [S]}$

- Non-competitive inhibition: $\displaystyle v = \frac{V_{max}}{1+[I]/K_i} \cdot \frac{[S]}{K_M + [S]}$

- Hill activation (switch ON): $\displaystyle v = \frac{V_{max}[S]^n}{K^n + [S]^n}$

- Hill repression (switch OFF): $\displaystyle v = \frac{V_{max} K^n}{K^n + [S]^n}$


In [ ]:
Vmax, KM, Ki = 10.0, 5.0, 2.0
S = np.linspace(0, 50, 400)

# Competitive inhibition: KM_app rises, Vmax stays fixed
def v_competitive(S, I):
    KM_app = KM * (1 + I / Ki)
    return Vmax * S / (KM_app + S)

# Non-competitive inhibition: Vmax_app falls, KM stays fixed
def v_noncompetitive(S, I):
    Vmax_app = Vmax / (1 + I / Ki)
    return Vmax_app * S / (KM + S)

I_values = [0, 2, 6, 20]
for I in I_values:
    print(f"I={I:5.1f}: competitive KM_app={KM * (1 + I / Ki):5.2f}, Vmax_app={Vmax:5.2f}"
          f"  |  non-competitive KM_app={KM:5.2f}, Vmax_app={Vmax / (1 + I / Ki):5.2f}")


In [ ]:
# Hill function: activation (rising sigmoid) and repression (falling sigmoid)
def hill_activation(S, n, K=KM):
    return Vmax * S**n / (K**n + S**n)

def hill_repression(S, n, K=KM):
    return Vmax * K**n / (K**n + S**n)

n_values = [1, 2, 4, 8]


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for I in I_values:
    axes[0, 0].plot(S, v_competitive(S, I), label=f"I={I}")
axes[0, 0].set_title("Competitive inhibition")
axes[0, 0].legend()

for I in I_values:
    axes[0, 1].plot(S, v_noncompetitive(S, I), label=f"I={I}")
axes[0, 1].set_title("Non-competitive inhibition")
axes[0, 1].legend()

for n in n_values:
    axes[1, 0].plot(S, hill_activation(S, n), label=f"n={n}")
axes[1, 0].set_title("Hill activation (soft switch ON)")
axes[1, 0].legend()

for n in n_values:
    axes[1, 1].plot(S, hill_repression(S, n), label=f"n={n}")
axes[1, 1].set_title("Hill repression (soft switch OFF)")
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel("[S]")
    ax.set_ylabel("v")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Python Exercises

### Exercise 1 *(CPMK2)*
At $[S] = K_M$, compute `v_competitive` and `v_noncompetitive` for $I = K_i$. Which one has dropped further from the uninhibited rate?


In [ ]:
# TODO: Exercise 1 — compute v_competitive(KM, Ki) and v_noncompetitive(KM, Ki), compare to v at I=0


### Exercise 2 *(CPMK2)*
For the Hill activation curve, find the value of $[S]$ where $v = V_{max}/2$ for each $n$ in `n_values`. Does it depend on $n$?


In [ ]:
# TODO: Exercise 2 — for each n in n_values, find S where hill_activation(S, n) crosses Vmax/2 (it should always be K)


### Exercise 3 *(CPMK2)*
Compute the ratio $[S]_{90\%}/[S]_{10\%}$ (substrate needed to go from 10% to 90% of $V_{max}$) for $n=1$ and $n=8$ on the Hill activation curve. How does it quantify "switch-like" behavior?


In [ ]:
# TODO: Exercise 3 — solve hill_activation(S, n) = 0.1*Vmax and = 0.9*Vmax for n=1 and n=8, report the ratio S_90/S_10


## Discussion Questions

1. Why does competitive inhibition become negligible at very high $[S]$, but non-competitive inhibition does not?
2. Biologically, what kind of binding process would give a Hill coefficient $n > 1$?
3. Why is a sigmoidal ("soft switch") response often more useful for a cell than a purely hyperbolic Michaelis-Menten response?

## Reading

- Ingalls (2013) Chapter 3: enzyme inhibition, cooperativity, and the Hill equation.
- Alon (2006) Chapter 2: the Hill function as an activator/repressor input function.
- Swain, PSB notes (enzyme kinetics section). [notes.pdf](https://swainlab.bio.ed.ac.uk/psb/lectures/notes.pdf) · [course site](https://swainlab.bio.ed.ac.uk/psb/index.html)
